# C-MET Colab 演示复现

这个 Colab 笔记本用来运行 C-MET 官方推理演示，对应论文 **Cross-Modal Emotion Transfer for Emotion Editing in Talking Face Video**。

优先使用 CUDA 运行时：`A100 GPU > L4 GPU > T4 GPU`。官方 `inference.py` 里有直接 `.cuda()` 调用，所以 CPU/Mac/MPS 不适合作为第一次复现目标。

目标：先生成一个基础情绪视频和一个扩展情绪视频，再考虑训练。

## 1. 检查 GPU

先确认 Colab 真的分配到了 GPU，再开始跑耗时步骤。

In [ ]:
!nvidia-smi
!python --version

## 2. 克隆官方 C-MET

这个复现仓库保持轻量，不直接存官方 C-MET 源码或权重检查点。

In [ ]:
!rm -rf C-MET
!git clone https://github.com/ChanHyeok-Choi/C-MET.git
%cd C-MET

## 3. 安装系统依赖

官方 README 建议通过 conda 安装 `ffmpeg=4.4`。在 Colab 里，apt 安装的 ffmpeg 通常足够跑第一次演示。

In [ ]:
!apt-get update -y
!apt-get install -y ffmpeg pkg-config
!ffmpeg -version | head -n 3

## 4. 安装最小 Python 依赖

官方 `requirements.txt` 很重，包含训练、TTS、ASR、Gradio 和超分包，在 Colab 的 Python 3.12 上容易失败。第一次演示只安装推理需要的最小依赖，并保持 Colab 核心包版本兼容；先不要加 `--sr`。

In [ ]:
!python -m pip install --no-cache-dir \
  'setuptools<82' 'jedi>=0.16' \
  huggingface_hub==1.20.1 omegaconf==2.3.0 tqdm==4.67.3 \
  pandas==2.2.2 scipy==1.16.3 \
  librosa==0.10.2.post1 'soundfile>=0.12.1' \
  imageio==2.37.3 imageio-ffmpeg==0.6.0 \
  opencv-python-headless==4.13.0.92 pillow==11.3.0

## 4.1 修补可选导入和兼容问题

官方脚本导入了 `funasr`，但我们提供了预先抽取好的 `emotion2vec+large` 特征，所以演示路径不需要它。Colab 还可能遇到旧 `librosa`/NumPy 别名问题、`moviepy.editor` 在 Python 3.12 上的兼容问题、新版 PyTorch 读取权重检查点的 `weights_only` 问题，以及当前 torchvision 缺少旧版 `read_video`/`write_video` 接口的问题。这个单元会把这些坑一次性补上。

In [ ]:
from pathlib import Path
import re

path = Path("inference.py")
text = path.read_text()
text = re.sub(
    r"(?m)^from funasr import AutoModel$",
    "try:\n    from funasr import AutoModel\nexcept Exception:\n    AutoModel = None",
    text,
)
path.write_text(text)

audio_path = Path("src/audio.py")
audio_text = audio_path.read_text()
audio_text = audio_text.replace(
    "import librosa\nimport librosa.filters\nimport numpy as np\n",
    "import numpy as np\n"
    "if not hasattr(np, 'complex'):\n    np.complex = complex\n"
    "if not hasattr(np, 'float'):\n    np.float = float\n"
    "if not hasattr(np, 'int'):\n    np.int = int\n"
    "import librosa\nimport librosa.filters\n",
)
audio_path.write_text(audio_text)

for moviepy_path in [Path("inference.py"), Path("src/util.py")]:
    moviepy_text = moviepy_path.read_text()
    moviepy_text = moviepy_text.replace(
        "from moviepy.editor import *\n",
        "# moviepy.editor 只用于可选的 --sr 超分后处理。\n"
        "VideoFileClip = None\n"
        "AudioFileClip = None\n",
    )
    moviepy_path.write_text(moviepy_text)

text = path.read_text()
text = text.replace(
    "torch.load(audio2lip_model_path, map_location=lambda storage, loc: storage)",
    "torch.load(audio2lip_model_path, weights_only=False, map_location=lambda storage, loc: storage)",
)
text = text.replace(
    "torch.load(pretrained_EDTalk['model_path'], map_location=lambda storage, loc: storage)",
    "torch.load(pretrained_EDTalk['model_path'], weights_only=False, map_location=lambda storage, loc: storage)",
)
text = text.replace(
    "torch.load(connector_exp_path, map_location=lambda storage, loc: storage)",
    "torch.load(connector_exp_path, weights_only=False, map_location=lambda storage, loc: storage)",
)
path.write_text(text)

marker = "# --- COLAB_VIDEO_IO_PATCH ---"
util_path = Path("src/util.py")
util_text = util_path.read_text().split(marker)[0].rstrip() + "\n\n"
video_patch = r'''
# --- COLAB_VIDEO_IO_PATCH ---
def vid_preprocessing(vid_path):
    import imageio.v2 as imageio
    import numpy as np
    import torch
    import torch.nn.functional as F

    reader = imageio.get_reader(vid_path, "ffmpeg")
    meta = reader.get_meta_data()
    fps = meta.get("fps", 25)
    frames = []
    try:
        for frame in reader:
            if frame.ndim == 2:
                frame = np.repeat(frame[..., None], 3, axis=2)
            if frame.shape[-1] == 4:
                frame = frame[..., :3]
            frames.append(frame)
    finally:
        reader.close()

    if not frames:
        raise ValueError(f"没有从视频中解码出帧：{vid_path}")

    arr = np.stack(frames).astype(np.float32)
    vid = torch.from_numpy(arr).permute(0, 3, 1, 2).unsqueeze(0)
    vid_norm = (vid / 255.0 - 0.5) * 2.0
    b, t, c, h, w = vid_norm.shape
    resized = F.interpolate(
        vid_norm.reshape(b * t, c, h, w),
        size=(256, 256),
        mode="bilinear",
        align_corners=False,
    )
    return resized.reshape(b, t, c, 256, 256), fps


def save_video(vid_target_recon, save_path, fps):
    import os
    import imageio.v2 as imageio
    import torch

    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    vid = vid_target_recon.detach().permute(0, 2, 3, 4, 1).clamp(-1, 1).cpu()
    vid = ((vid + 1.0) * 127.5).clamp(0, 255).to(torch.uint8).numpy()[0]
    writer = imageio.get_writer(
        save_path,
        fps=float(fps),
        codec="libx264",
        quality=8,
        macro_block_size=1,
    )
    try:
        for frame in vid:
            writer.append_data(frame)
    finally:
        writer.close()
'''
util_path.write_text(util_text + video_patch)

print("兼容补丁已完成：可选导入、权重检查点加载、imageio 视频读写")

## 4.2 验证补丁是否生效

这个单元会在长时间推理前提前检查 `np.complex`/`librosa`、`torch.load` 和 `torchvision.io.read_video` 这些常见问题。

In [ ]:
import numpy as np
import librosa
from pathlib import Path
print("numpy:", np.__version__)
print("librosa:", librosa.__version__)
txt = Path("src/util.py").read_text()
infer = Path("inference.py").read_text()
assert "COLAB_VIDEO_IO_PATCH" in txt
assert "imageio.get_reader" in txt
assert "imageio.get_writer" in txt
assert "weights_only=False" in infer
print("兼容补丁：通过")

## 5. 验证 PyTorch CUDA

In [ ]:
import torch
print("torch:", torch.__version__)
print("CUDA 是否可用:", torch.cuda.is_available())
print("GPU 设备:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "无")

## 6. 运行基础情绪演示：happy

第一次运行可能会从 Hugging Face 下载 C-MET 和 EDTalk 权重。这里先不要加 `--sr`。

In [ ]:
!python inference.py --num_samples 10 \
  --connector_exp_path ./checkpoints/_epoch_2105_checkpoint_step000200000.pth \
  --source_path ./asset/identity/ChatGPT_man3_crop.png \
  --audio_driving_path ./asset/audio/W009_038.wav \
  --pose_driving_path ./asset/video/W009_038.mp4 \
  --save_path ./res/ChatGPT_man3_happy.mp4 \
  --neu_e2v_path ./audios/MEAD/neutral/emotion2vec+large_features/ \
  --emo_e2v_path ./audios/MEAD/happy/emotion2vec+large_features/

## 7. 显示 happy 结果

In [ ]:
from IPython.display import Video, display
display(Video("./res/ChatGPT_man3_happy.mp4", embed=True))

## 8. 运行扩展情绪演示：sarcastic

这个例子更适合汇报，因为论文强调了扩展情绪。

In [ ]:
!python inference.py --num_samples 10 \
  --connector_exp_path ./checkpoints/_epoch_2105_checkpoint_step000200000.pth \
  --source_path ./asset/identity/ChatGPT_man3_crop.png \
  --audio_driving_path ./asset/audio/W009_038.wav \
  --pose_driving_path ./asset/video/W009_038.mp4 \
  --save_path ./res/ChatGPT_man3_sarcastic.mp4 \
  --neu_e2v_path ./audios/MEAD/neutral/emotion2vec+large_features/ \
  --emo_e2v_path ./audios/gemini/sarcastic/emotion2vec+large_features/

## 9. 显示 sarcastic 结果

In [ ]:
display(Video("./res/ChatGPT_man3_sarcastic.mp4", embed=True))

## 10. 评估输出：技术检查

这是第一层评估：确认生成文件是有效视频，有音频、分辨率正确、帧率和时长正常。

In [ ]:
import json
import subprocess
from pathlib import Path

outputs = {
    "happy": Path("./res/ChatGPT_man3_happy.mp4"),
    "sarcastic": Path("./res/ChatGPT_man3_sarcastic.mp4"),
}

for label, path in outputs.items():
    if not path.exists():
        raise FileNotFoundError(path)
    info = json.loads(subprocess.check_output([
        "ffprobe",
        "-v", "error",
        "-show_entries", "format=duration,size:stream=codec_type,codec_name,width,height,avg_frame_rate,nb_frames",
        "-of", "json",
        str(path),
    ], text=True))
    streams = info.get("streams", [])
    video = next((s for s in streams if s.get("codec_type") == "video"), {})
    audio = next((s for s in streams if s.get("codec_type") == "audio"), None)
    fps_text = video.get("avg_frame_rate", "0/1")
    num, den = [float(x) for x in fps_text.split("/")]
    fps = num / den if den else 0.0
    duration = float(info.get("format", {}).get("duration", 0.0))
    size_mb = int(info.get("format", {}).get("size", 0)) / (1024 * 1024)
    print(f"[{label}] {path}")
    print(f"  分辨率: {video.get('width')}x{video.get('height')}")
    print(f"  FPS: {fps:.2f}, 帧数: {video.get('nb_frames', '未知')}, 时长: {duration:.2f} 秒")
    print(f"  视频编码: {video.get('codec_name')}, 音频编码: {audio.get('codec_name') if audio else '缺失'}")
    print(f"  文件大小: {size_mb:.2f} MB")
    assert video.get("width") == 256 and video.get("height") == 256
    assert duration > 0
    assert audio is not None, "没有检测到音频流"
print("技术检查：通过")

## 10.1 评估输出：帧截图表

这个图可以放进汇报里，用来展示时间一致性。重点看：身份有没有漂移、嘴型有没有崩、画面是否严重模糊、是否闪烁，以及 happy 和 sarcastic 的表情是否有明显差异。

In [ ]:
import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(len(outputs), 5, figsize=(15, 5))
if len(outputs) == 1:
    axes = np.array([axes])

for row, (label, path) in enumerate(outputs.items()):
    reader = imageio.get_reader(str(path), "ffmpeg")
    meta = reader.get_meta_data()
    fps = float(meta.get("fps", 25))
    duration = float(meta.get("duration", 0) or 0)
    try:
        total_frames = reader.count_frames()
    except Exception:
        total_frames = max(1, int(duration * fps))
    indices = np.linspace(0, max(total_frames - 1, 0), 5, dtype=int)
    try:
        for col, idx in enumerate(indices):
            frame = reader.get_data(int(idx))
            axes[row, col].imshow(frame)
            axes[row, col].set_title(f"{label}\n第 {idx} 帧")
            axes[row, col].axis("off")
    finally:
        reader.close()

plt.tight_layout()
plt.show()

## 10.2 汇报用评估表

| 维度 | 检查什么 | 成功信号 |
|---|---|---|
| 技术有效性 | MP4 能播放、有音频、256x256、时长大于 0 | `技术检查：通过` |
| 情绪迁移 | happy 应该更积极；sarcastic 应该和 neutral/happy 有差异 | 抽样帧里能看到表情变化 |
| 身份保持 | 是否还是同一个人，脸型和纹理是否稳定 | 没有明显身份漂移或脸崩 |
| 唇形同步 | 嘴部运动是否跟随驱动音频 | 没有明显音画延迟 |
| 姿态和运动一致性 | 头部姿态是否平滑跟随驱动视频 | 没有严重抖动或冻结帧 |
| 视觉质量 | 脸部区域是否连贯 | 没有严重模糊、撕裂、边缘伪影或闪烁 |

一周内的复现，最低可汇报结论是：官方权重检查点可以在 Colab 上跑通，并生成有效的 happy/sarcastic talking-face 视频；视频里身份、音频和面部运动大体连贯。更强的量化汇报可以再补 SyncNet 唇音同步分数、ArcFace 身份相似度，以及抽样帧的人脸情绪分类分数。

## 11. 保存输出到 Google Drive

这是可选步骤，但建议在 Colab 断开前保存结果。

In [ ]:
# 可选：如果你想把结果保存到 Google Drive，就取消下面几行注释。
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/C-MET-results
# !cp ./res/*.mp4 /content/drive/MyDrive/C-MET-results/

## 什么算成功

- `./res/ChatGPT_man3_happy.mp4` 存在并且能播放。
- `./res/ChatGPT_man3_sarcastic.mp4` 存在并且能播放。
- 评估单元打印 `技术检查：通过`。
- 帧截图表显示身份稳定、嘴型基本合理，并且两种情绪风格有明显差异。
- 你记录了 GPU 型号、运行命令和遇到的报错。

这些跑通后，下一步是小样本训练，不是直接做完整 MEAD 训练。